# 00 — NASA API Discovery

**Goal:** Use NASA's open data catalog (CKAN) and Zenodo APIs to locate the AI4Mars dataset metadata and download links.

We will **not** download the full dataset in this notebook — that step is manual to avoid committing large binary files to the repository.

---

## Why two APIs?

| Portal | Purpose |
|---|---|
| **data.nasa.gov** | NASA's open-data metadata catalog (CKAN). Good for *discovering* datasets, but does not always host the actual files. |
| **zenodo.org** | Open research data repository. The AI4Mars files (~14 GB) are hosted here as a **Zenodo record**. |

The workflow is:
1. Use NASA CKAN to find the dataset name / description.
2. Use Zenodo to get the exact file list and download URLs.
3. Download manually (or with `wget`/`curl`) and extract into `data/raw/`.

In [ ]:
import requests
import pandas as pd
from pprint import pprint

# Add the project root to the Python path so we can import from src/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.nasa_catalog import (
    search_nasa_datasets,
    get_nasa_dataset,
    get_zenodo_record,
    list_zenodo_files,
    print_nasa_search_results,
    print_zenodo_files,
)

## Step 1 — Search NASA Data Catalog

In [ ]:
# Search NASA CKAN for AI4Mars-related datasets
search_response = search_nasa_datasets("AI4Mars Mars terrain autonomous driving", rows=10)

# Pretty-print the titles and dataset names
print_nasa_search_results(search_response)

## Step 2 — Inspect a Specific NASA Dataset

If you found an AI4Mars dataset in the search above, replace `dataset_id` below with the exact name from the `name` field.

In [ ]:
# Replace with the actual dataset name from the search results above.
# This may fail if NASA CKAN doesn't have a dedicated AI4Mars entry.
dataset_id = "ai4mars-a-dataset-for-terrain-aware-autonomous-driving-on-mars"

try:
    dataset_info = get_nasa_dataset(dataset_id)
    print(f"Title      : {dataset_info['result'].get('title', 'N/A')}")
    print(f"Description: {dataset_info['result'].get('notes', 'N/A')[:300]}...")
except Exception as e:
    print(f"Could not load dataset '{dataset_id}': {e}")
    print("This is expected if NASA CKAN does not have a direct entry — proceed to Zenodo below.")

## Step 3 — Zenodo Record Metadata

The AI4Mars dataset is published on Zenodo.
The record ID is **4033453** — you can verify it at https://zenodo.org/record/4033453

In [ ]:
# Zenodo record ID for AI4Mars
# Change this if a newer version has been published.
ZENODO_RECORD_ID = 4033453

record = get_zenodo_record(ZENODO_RECORD_ID)

print(f"Title      : {record.get('metadata', {}).get('title', 'N/A')}")
print(f"Description: {record.get('metadata', {}).get('description', 'N/A')[:400]}...")
print(f"DOI        : {record.get('doi', 'N/A')}")
print(f"Published  : {record.get('metadata', {}).get('publication_date', 'N/A')}")

## Step 4 — List Zenodo Files

This shows the individual files in the Zenodo record, their sizes, and direct download URLs.

In [ ]:
files = list_zenodo_files(record)
print(f"Found {len(files)} file(s) in Zenodo record {ZENODO_RECORD_ID}:\n")
print_zenodo_files(files)

In [ ]:
# Display as a tidy DataFrame for easier reading
df = pd.DataFrame([
    {
        "filename": f.get("key"),
        "size_mb": round(f.get("size", 0) / 1_000_000, 1),
        "download_url": f.get("links", {}).get("self", "N/A"),
    }
    for f in files
])
df

## Next Steps

1. Copy the download URL(s) from the table above.
2. Download the archive manually (or use `wget`/`curl` in a terminal) into `data/raw/`.
3. Extract the archive: `unzip ai4mars.zip -d data/raw/`
4. Open `01_dataset_inspection.ipynb` to verify the extracted structure.

> **Tip:** The full dataset is large (~14 GB). Start with a small subset if available.